# Week 11 skills notebook: pattern or chance

This notebook belongs to the skills track. Every dataset in it is made up with a seed, so you know the truth in advance and can see when a method finds it and when a method is fooled.

You will run a permutation test by hand, read its p-value in one honest sentence, see why a large sample makes a small difference look important, test a trend, learn what a shuffle assumes about time, and see what happens when you try many comparisons and keep the best one. The formula comes last, as a shortcut for a test you already understand.

**How to use it:** run each cell in order with the play button or Shift+Enter. Read the output before you move on.

## 1. The question every test asks

A farmer tests a new fertiliser on 24 plots of the same crop. She picks 12 plots at random for the fertiliser and leaves the other 12 as they are. At harvest she weighs the yield of each plot in kilograms.

The fertilised plots yield more on average. A gap like that can come from the fertiliser, or from the luck of which plots happened to get it. Every test of a pattern asks the same question: **if the fertiliser did nothing, how often would chance alone produce a gap this large?**

The data below were made with a true effect of 4 kg, so you know the right answer is "the fertiliser helps".

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(28)
control = rng.normal(40, 5, 12).round(1)      # 12 plots without fertiliser, kg
treated = rng.normal(44, 5, 12).round(1)      # 12 plots with fertiliser, kg

gap = treated.mean() - control.mean()
print("Mean yield without fertiliser (kg):", round(control.mean(), 2))
print("Mean yield with fertiliser (kg):", round(treated.mean(), 2))
print("Gap (kg):", round(gap, 2))

## 2. One shuffle

If the fertiliser did nothing, the labels "fertilised" and "not fertilised" are only names stuck on 24 yields. Any other way of sticking 12 of each label on them was just as likely.

A **shuffle** tests that idea. Pool the 24 yields, deal them out at random into two groups of 12, and compute the gap again. The cell below deals five shuffles.

In [ ]:
pooled = np.concatenate([treated, control])
shuffler = np.random.default_rng(1)
for i in range(1, 6):
    shuffled = shuffler.permutation(pooled)
    print("shuffle", i, "gap (kg):", round(shuffled[:12].mean() - shuffled[12:].mean(), 2))

## 3. Ten thousand shuffles

Five shuffles show that chance alone makes gaps of a few kilograms in either direction. Ten thousand shuffles show how often chance makes a gap as large as the real one.

The **p-value** is the share of shuffles whose gap is at least as large as the real gap, in either direction. The test counts both directions because before the harvest nobody knew whether the fertiliser would help or harm.

The function `shuffle_gaps` does the work. You will use it again in later sections.

In [ ]:
def shuffle_gaps(a, b, n_shuffles, rng):
    """The gap between two groups after each of n_shuffles random relabellings."""
    pooled = np.concatenate([a, b])
    gaps = np.empty(n_shuffles)
    for i in range(n_shuffles):
        s = rng.permutation(pooled)
        gaps[i] = s[:len(a)].mean() - s[len(a):].mean()
    return gaps

gaps = shuffle_gaps(treated, control, 10000, np.random.default_rng(2))
as_large = int(np.sum(np.abs(gaps) >= abs(gap)))
p_value = as_large / 10000
print("Shuffles with a gap at least as large as the real one:", as_large, "of 10000")
print("p-value:", round(p_value, 4))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(gaps, bins=60, color="#999999")
ax.axvline(gap, color="#c0392b", label="real gap")
ax.axvline(-gap, color="#c0392b", linestyle="--", label="same size, other direction")
ax.set_title("Gaps from 10,000 shuffles of the fertiliser labels")
ax.set_xlabel("gap in mean yield (kg)")
ax.set_ylabel("shuffles")
ax.legend()
plt.tight_layout()
plt.show()

## 4. One honest sentence

A p-value answers one question and only that one. The cell below writes it as a sentence with your numbers filled in.

Two other readings are wrong, and both are common.

- **The p-value is not the chance that the fertiliser does nothing.** The test assumes the fertiliser does nothing and asks how surprising the data would be under that assumption.
- **The p-value is not the size of the effect.** A small p-value can come with a gap too small to matter, as section 5 shows.

Many fields call a result "statistically significant" when the p-value is below 0.05. That cut-off is a convention. A result at 0.049 and a result at 0.051 carry almost the same evidence.

In [ ]:
print(f"If the fertiliser did nothing, a random split of the plots would give a gap of at least "
      f"{abs(gap):.2f} kg in {p_value:.1%} of splits.")

## 5. Significant and small

A large sample can make a tiny difference look important. The next cell compares two seed varieties on 20,000 plots each. The data were made with a true difference of 0.3 kg on plots that yield about 40 kg.

When no shuffle reaches the real gap, the p-value is below one divided by the number of shuffles. A p-value is never exactly zero, so never write p = 0.

In [ ]:
rng = np.random.default_rng(5)
variety_a = rng.normal(40.0, 5, 20000)
variety_b = rng.normal(40.3, 5, 20000)
big_gap = variety_b.mean() - variety_a.mean()
big_gaps = shuffle_gaps(variety_b, variety_a, 1000, np.random.default_rng(6))
print("Gap (kg):", round(big_gap, 3))
print("Shuffles with a gap at least as large:", int(np.sum(np.abs(big_gaps) >= abs(big_gap))), "of 1000")
print("Gap as a share of the mean yield:", f"{big_gap / variety_a.mean():.2%}")
print("Gap in standard deviations:", round(big_gap / variety_a.std(), 3))

The gap is real, because the data were made with it, and it is too small to matter to a farmer. The number that answers "does it matter?" is the **effect size**: the gap in kilograms, the gap as a share of the mean, or the gap in standard deviations. Report the effect size next to every p-value.

## 6. The same effect at three sample sizes

The p-value depends on two things: how large the effect is and how much data you have. The cell below makes data with the same true effect of 2 kg at three sample sizes.

In [ ]:
for n in [10, 100, 1000]:
    rng = np.random.default_rng(n)
    a = rng.normal(40, 5, n)
    b = rng.normal(42, 5, n)
    g = b.mean() - a.mean()
    p = np.mean(np.abs(shuffle_gaps(b, a, 2000, np.random.default_rng(7))) >= abs(g))
    print(f"{n} plots per group: gap {g:.2f} kg, p-value {p:.4f}")

The true effect never changes. The estimate of it settles near 2 kg as the sample grows, and the p-value falls. A large p-value from a small sample means "this sample cannot tell the effect from chance". It does not mean "there is no effect".

## 7. A trend over time

A trend is a pattern in time, and the shuffle works for it too. The made-up table below holds 30 years of visitors to a nature park, in thousands. The slope of a straight line through the points is the trend, in thousands of visitors per year. The data were made with no trend at all.

To test a slope, shuffle the order of the values. If there is no trend, any order of the values was as likely as the real one.

In [ ]:
rng = np.random.default_rng(59)
years = np.arange(1996, 2026)
visitors = (50 + rng.normal(0, 4, 30)).round(1)     # thousands, made with no trend
park = pd.DataFrame({"year": years, "visitors_k": visitors})
print(park.tail(5))

We gave an AI assistant the last five rows and asked: *"Is the number of visitors rising?"* It wrote the cell below. Run it.

In [ ]:
# Written by an AI assistant
recent = park.tail(5)
slope = np.polyfit(recent["year"], recent["visitors_k"], 1)[0]
print("Clear upward trend: visitors grow by", round(slope, 2), "thousand a year")

The arithmetic is right. The claim is the problem, because five values can lean upward by luck. Five values have only 120 possible orders, so the test can try every one of them instead of shuffling at random. Then the same test runs on all 30 years.

In [ ]:
import itertools

def slope_of(values, x):
    return np.polyfit(x, values, 1)[0]

x5 = recent["year"].to_numpy()
orders = np.array([slope_of(np.array(order), x5) for order in itertools.permutations(recent["visitors_k"])])
p_five = np.mean(np.abs(orders) >= abs(slope) - 1e-9)
print("Orders of the five values:", len(orders))
print("Five-year p-value:", round(p_five, 3))

all_slope = slope_of(visitors, years)
shuffler = np.random.default_rng(8)
all_slopes = np.array([slope_of(shuffler.permutation(visitors), years) for _ in range(5000)])
p_all = np.mean(np.abs(all_slopes) >= abs(all_slope))
print("Slope over all 30 years (thousand a year):", round(all_slope, 3))
print("Thirty-year p-value:", round(p_all, 3))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(years, visitors, "o", color="#999999")
ax.plot(x5, np.polyval(np.polyfit(x5, recent["visitors_k"], 1), x5), color="#c0392b", label="line through the last five years")
ax.plot(years, np.polyval(np.polyfit(years, visitors, 1), years), color="#2c3e50", label="line through all 30 years")
ax.set_title("Park visitors: five years against thirty")
ax.set_xlabel("year")
ax.set_ylabel("visitors (thousands)")
ax.legend()
plt.tight_layout()
plt.show()

Compare the two p-values. Five years cannot rule out luck, and the full record shows no trend, which is the truth the data were made with. Before you accept a trend, ask how many years it rests on and how often that many values in random order would lean as steeply.

## 8. What the shuffle assumes

A shuffle treats every order of the data as equally likely when there is no pattern. That holds when each value is a separate draw, like plots picked at random. It fails when the data run in time and each week sits close to the week before.

The two made-up weekly series below have no link to each other. Each one drifts slowly up and down on its own, the way prices, river levels and case counts often do.

In [ ]:
def drift(rng, n, keep=0.95):
    """A series in which each week keeps most of last week's value and adds some noise."""
    x = np.zeros(n)
    noise = rng.normal(0, 1, n)
    for t in range(1, n):
        x[t] = keep * x[t - 1] + noise[t]
    return x

rng = np.random.default_rng(128)
series_a = 100 + 5 * drift(rng, 300)
series_b = 50 + 3 * drift(rng, 300)
r = np.corrcoef(series_a, series_b)[0, 1]
print("Correlation between the two unrelated series:", round(r, 3))

fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].plot(series_a)
ax[0].set_ylabel("series A")
ax[0].set_title("Two made-up series with no link between them")
ax[1].plot(series_b, color="#c0392b")
ax[1].set_ylabel("series B")
ax[1].set_xlabel("week")
plt.tight_layout()
plt.show()

An ordinary shuffle mixes the weeks of series B into a random order. That destroys the slow drift, so the shuffled copies look nothing like a real drifting series.

In [ ]:
shuffler = np.random.default_rng(9)
shuffled_r = np.array([np.corrcoef(series_a, shuffler.permutation(series_b))[0, 1] for _ in range(5000)])
p_ordinary = np.mean(np.abs(shuffled_r) >= abs(r))
print("Ordinary shuffle: shuffles at least as strong:", int(np.sum(np.abs(shuffled_r) >= abs(r))), "of 5000")

The ordinary shuffle says a correlation this strong almost never happens by chance, and it is wrong, because the series were made with no link. A **shift test** keeps each series in its own order and slides one past the other. Every shift keeps the drift, and only the alignment between the two series changes. `np.roll` moves the values that fall off the end back to the start. Shifts shorter than half a year (26 weeks) are left out, so that no week lines up with itself or its near neighbours.

In [ ]:
shifts = range(26, len(series_b) - 25)
shifted_r = np.array([np.corrcoef(series_a, np.roll(series_b, s))[0, 1] for s in shifts])
p_shift = np.mean(np.abs(shifted_r) >= abs(r))
print("Shifts tried:", len(shifted_r))
print("Shift test p-value:", round(p_shift, 3))

Two slowly drifting series often line up by accident over a few hundred weeks. The shift test allows for that and the ordinary shuffle does not. Before you shuffle, ask whether the order of your data carries information. If it does, shuffle in a way that keeps the order.

## 9. Many tries, one winner

A team records 20 habits of 40 made-up students (hours of sleep, minutes of reading and so on) next to their exam scores. The data were made so that no habit affects the score. The team tests every habit and reports the one with the smallest p-value.

The honest test repeats the whole search inside every shuffle. It asks how often the best of 20 useless habits looks as strong as the winner did.

In [ ]:
rng = np.random.default_rng(68)
score = rng.normal(0, 1, 40)
habits = rng.normal(0, 1, (40, 20))       # 20 habits, none linked to the score

def corr_with(y, X):
    """The correlation of y with each column of X."""
    yc = (y - y.mean()) / y.std()
    Xc = (X - X.mean(axis=0)) / X.std(axis=0)
    return Xc.T @ yc / len(y)

real_r = corr_with(score, habits)
best = int(np.argmax(np.abs(real_r)))
print("Habit with the strongest correlation:", best, "with r =", round(real_r[best], 3))

shuffler = np.random.default_rng(10)
one_habit, best_of_all = [], []
for _ in range(5000):
    r_shuffled = corr_with(shuffler.permutation(score), habits)
    one_habit.append(abs(r_shuffled[best]))
    best_of_all.append(np.abs(r_shuffled).max())
p_one = np.mean(np.array(one_habit) >= abs(real_r[best]))
p_search = np.mean(np.array(best_of_all) >= abs(real_r[best]))
print("p-value as if this habit were the only one tested:", round(p_one, 4))
print("p-value for the best of 20 habits:", round(p_search, 4))
print("False alarms to expect among 20 useless habits at the 0.05 cut-off:", 20 * 0.05)

Report how many comparisons you tried, and correct for them. A result that survives only when you hide the other 19 tries is a result chance can produce.

## 10. The formula comes second

Statisticians found formulas that predict the shuffle result without running it. The best known is the t-test. It assumes the two groups are separate random samples. On the fertiliser data it gives a p-value close to the one the shuffles gave.

The same kind of formula, used on the drifting series from section 8, is fooled in the same way the ordinary shuffle was. A formula is a fast shortcut for a shuffle, and it carries the shuffle's assumptions with it.

In [ ]:
from scipy import stats

t_result = stats.ttest_ind(treated, control, equal_var=False)
print("Shuffle test p-value, fertiliser:", round(p_value, 4))
print("t-test p-value, fertiliser:", round(t_result.pvalue, 4))
print("Formula p-value, two unrelated drifting series:", f"{stats.pearsonr(series_a, series_b).pvalue:.1e}")
print("Shift test p-value, the same series:", round(p_shift, 3))

## 11. Your turn: two delivery routes

A delivery firm times 15 trips on route A and 15 trips on route B, in minutes. Ask your AI assistant for code that runs a permutation test on the gap in mean time (route B minus route A) with at least 10,000 shuffles. Store the gap in `gap_route` and the p-value in `p_route`, then run the CHECK cell.

Before you trust the code, check two things. Does it shuffle all 30 trips together? Does it count gaps in both directions?

In [ ]:
rng = np.random.default_rng(37)
route_a = rng.normal(52, 8, 15).round(1)
route_b = rng.normal(57, 8, 15).round(1)
print("Route A mean (minutes):", round(route_a.mean(), 2))
print("Route B mean (minutes):", round(route_b.mean(), 2))

In [ ]:
# your turn 1: store the gap (route B minus route A) in gap_route and the p-value in p_route
gap_route = None
p_route = None

In [ ]:
# CHECK 1
assert gap_route is not None and p_route is not None, "gap_route or p_route is still empty"
assert abs(gap_route - (route_b.mean() - route_a.mean())) < 1e-6, "the gap should be route B minus route A"
reference = shuffle_gaps(route_b, route_a, 50000, np.random.default_rng(99))
p_reference = np.mean(np.abs(reference) >= abs(gap_route))
assert abs(p_route - p_reference) < 0.02, "the p-value is off: does the code count both directions?"
print("CHECK 1 passed: gap", round(gap_route, 2), "minutes, p-value close to", round(p_reference, 3))

## 12. Your turn: is the wait getting longer?

A made-up clinic recorded the average wait, in minutes, for each of 20 years. Ask your AI assistant for a permutation test on the slope. Store the slope in minutes per year in `wait_slope` and the p-value from at least 10,000 shuffles of the waits in `p_wait`. Reuse `slope_of` from section 7 if you like.

In [ ]:
rng = np.random.default_rng(13)
wait_years = np.arange(2006, 2026)
wait = (35 + 0.4 * (wait_years - 2006) + rng.normal(0, 4, 20)).round(1)
print(pd.DataFrame({"year": wait_years, "wait_minutes": wait}).tail())

In [ ]:
# your turn 2: store the slope in wait_slope and the p-value in p_wait
wait_slope = None
p_wait = None

In [ ]:
# CHECK 2
assert wait_slope is not None and p_wait is not None, "wait_slope or p_wait is still empty"
assert abs(wait_slope - np.polyfit(wait_years, wait, 1)[0]) < 1e-6, "the slope should come from a straight line through all 20 years"
checker = np.random.default_rng(97)
reference = np.array([np.polyfit(wait_years, checker.permutation(wait), 1)[0] for _ in range(20000)])
p_reference = np.mean(np.abs(reference) >= abs(wait_slope))
assert abs(p_wait - p_reference) < 0.02, "the p-value is off: shuffle the waits, keep the years, count both directions"
print("CHECK 2 passed: slope", round(wait_slope, 3), "minutes a year, p-value close to", round(p_reference, 3))

## 13. Say it without overclaiming

Someone will always ask: "So is it true?" A test cannot answer that directly. It says how often chance alone would produce what you found, and nothing about why. Write every result twice: once for a specialist and once for anyone. The cell below does it for the fertiliser.

In [ ]:
print("For a specialist:")
print(f"  A permutation test with 10,000 shuffles gave p = {p_value:.3f} for a gap of {gap:.1f} kg.")
print("For anyone:")
print(f"  If the fertiliser did nothing, a gap this large would turn up in about {round(p_value * 100)} of every 100 random splits of the plots.")
print(f"  Chance is an unlikely explanation, though not an impossible one. The gap was {gap:.1f} kg on plots that yield about {control.mean():.0f} kg.")

Now write both versions for your delivery routes in a text cell below. The version for anyone must not contain the words "p-value" or "significant".